# WBDSinfo preparation

This notebook is written to create the WBDSinfo.xlsx file. It consists of a metadata sheet that will be uploaded in the updated version of the Figshare repository. The metadata sheet contains information about the datasets used in the WBDS project, including dataset names, descriptions, sources, and other relevant details.

In [1]:
import os
import warnings
from fnmatch import fnmatch
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

# scipy and numpy have too many future warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

pd.set_option('display.precision', 3)

## File directories
The compressed files contained in the WBDSascii.zip and WBDSinfo.xlsx should be dowloaded and located in the same directory.

In [2]:
fileDir     = Path(r'C:\Users\Reginaldo\Downloads\WBDSascii\51subjs')
fileDir_c3d = Path(r'C:\Users\Reginaldo\Downloads\WBDSc3d\WBDSc3d')
path2       = Path(r'./../data')

## Create metadata file that is consistent with the files published in the Figshare dataset.

In [3]:
# Import WBDSinfo2.csv. This file is a starting point but it lacks some information that is in the WBDSinfo.csv file. 
fname = path2 / 'WBDSinfo2.csv'
df_wbds = pd.read_csv(fname)

In [4]:
df_wbds.head()

,Subject,FileName,AgeGroup,Age,Height,Mass,Gender,Dominance,LegLength,Static1,Static2,GaitSpeed(m/s),TreadHands,FP_RightFoot,FP_LeftFoot,Notes,BorgScale,Unnamed: 17
0,1,WBDS01static1.c3d,Young,25,172.5,74.3,M,R,0.89,Yes,No,--,--,--,--,--,10.0,NaN
1,1,WBDS01walkT01.c3d,Young,25,172.5,74.3,M,R,0.89,Yes,No,0.49,No,--,--,--,10.0,NaN
2,1,WBDS01walkT02.c3d,Young,25,172.5,74.3,M,R,0.89,Yes,No,0.67,No,--,--,--,10.0,NaN
3,1,WBDS01walkT03.c3d,Young,25,172.5,74.3,M,R,0.89,Yes,No,0.85,No,--,--,--,10.0,NaN
4,1,WBDS01walkT04.c3d,Young,25,172.5,74.3,M,R,0.89,Yes,No,1.03,No,--,--,--,10.0,NaN


In [5]:
df_wbds.tail()

,Subject,FileName,AgeGroup,Age,Height,Mass,Gender,Dominance,LegLength,Static1,Static2,GaitSpeed(m/s),TreadHands,FP_RightFoot,FP_LeftFoot,Notes,BorgScale,Unnamed: 17
6991,51,WBDS51walkT04.c3d,Older,73,161.0,84.85,F,R,0.8,--,--,0.71,--,--,--,--,18.0,NaN
6992,51,WBDS51walkT05.c3d,Older,73,161.0,84.85,F,R,0.8,--,--,0.84,--,--,--,--,18.0,NaN
6993,51,WBDS51walkT06.c3d,Older,73,161.0,84.85,F,R,0.8,--,--,0.97,--,--,--,--,18.0,NaN
6994,51,WBDS51walkT07.c3d,Older,73,161.0,84.85,F,R,0.8,--,--,--,--,--,--,--,18.0,NaN
6995,51,WBDS51walkT08.c3d,Older,73,161.0,84.85,F,R,0.8,--,--,--,--,--,--,--,18.0,NaN


In [6]:
# drop 'Unnamed: 17' column
df_wbds = df_wbds.drop(columns=['Unnamed: 17'])

## Ensure both text and c3d files are present for each subject, and that the same amount (and names) of files are present.

In [7]:
# Get the filenames from fileDir and return a list of the filenames
def get_filenames(fileDir, ext='.txt'):
    filenames = []
    for file in os.listdir(fileDir):
        if file.endswith(ext):
            filenames.append(file)
    return filenames

In [8]:
filenames_txt = get_filenames(fileDir, ext='.txt')
filenames_c3d = get_filenames(fileDir_c3d, ext='.c3d')

In [9]:
# Filter filenames_txt to only include elements with either the pattern '*static.txt' or '*mkr.txt'
filenames_txt2 = [f for f in filenames_txt if fnmatch(f, '*static*.txt') or fnmatch(f, '*grf.txt')]

In [10]:
# Create two lists of names, one for the txt files and one for the c3d files, without the extensions. For the filenames_c3d list, remove the last 4 characters (the extension) and for the filenames_txt2 list, remove the last 7 characters if it doesn't contain 'static', otherwise remove the last 4 characters. Then, create a new list of names that are in both lists.
names_txt2 = [f[:-7] if 'static' not in f else f[:-4] for f in filenames_txt2]
names_c3d = [f[:-4] for f in filenames_c3d]

In [11]:
# Compare elements in names_txt2 and names_c3d and return a list of names that are not in both lists
names_not_in_both = [f for f in names_txt2 if f not in names_c3d] + [f for f in names_c3d if f not in names_txt2]
# print the results
print(f'Number of txt files: {len(names_txt2)}')
print(f'Number of c3d files: {len(names_c3d)}')
print(f'Number of files in both: {len(names_txt2) - len(names_not_in_both)}')
print(f'Files not in both: {names_not_in_both}')

Number of txt files: 2082
Number of c3d files: 2082
Number of files in both: 2082
Files not in both: []


## Create Metadata file based on the txt and c3d files in the Figshare dataset.

In [12]:
# Filter df_wbds to only include rows where the 'FileName' exists in filenames_txt or filenames_c3d
df_wbds_filtered = df_wbds[df_wbds['FileName'].isin(filenames_txt) | df_wbds['FileName'].isin(filenames_c3d)]

In [13]:
# Filter df_wbds to only include rows where the 'FileName' are not in df_wbds_filtered
df_wbds_not_in_filtered = df_wbds[~df_wbds['FileName'].isin(df_wbds_filtered['FileName'])]

Filenames that are present in the Figshare repository but are not listed in the 'WBDSinfo2.csv' file.

In [14]:
# print the filenames only in df_wbds_not_in_filtered
print(f'Number of files in df_wbds_not_in_filtered: {len(df_wbds_not_in_filtered)}')
# Output the list of filenames only in df_wbds_not_in_filtered
print(f'Files not listed in WBDSinfo2.csv: {df_wbds_not_in_filtered["FileName"].tolist()}')

Number of files in df_wbds_not_in_filtered: 104
Files not listed in WBDSinfo2.csv: ['WBDS03walkT02mkr.tx', 'WBDS03walkT03mkr.tx', 'WBDS03walkT04mkr.tx', 'WBDS03walkT05mkr.tx', 'WBDS03walkT06mkr.tx', 'WBDS03walkT07mkr.tx', 'WBDS03walkT08mkr.tx', 'WBDS03walkO01Smkr.tx', 'WBDS03walkO02Smkr.tx', 'WBDS03walkO03Smkr.tx', 'WBDS03walkO04Smkr.tx', 'WBDS03walkO05Smkr.tx', 'WBDS03walkO06Smkr.tx', 'WBDS03walkO07Smkr.tx', 'WBDS03walkO08Smkr.tx', 'WBDS03walkO09Smkr.tx', 'WBDS03walkO10Smkr.tx', 'WBDS03walkO11Smkr.tx', 'WBDS03walkO12Smkr.tx', 'WBDS03walkO13Smkr.tx', 'WBDS03walkO14Smkr.tx', 'WBDS03walkO01Cmkr.tx', 'WBDS03walkO02Cmkr.tx', 'WBDS03walkO03Cmkr.tx', 'WBDS03walkO04Cmkr.tx', 'WBDS03walkO05Cmkr.tx', 'WBDS03walkO06Cmkr.tx', 'WBDS03walkO07Cmkr.tx', 'WBDS03walkO08Cmkr.tx', 'WBDS03walkO01Fmkr.tx', 'WBDS03walkO02Fmkr.tx', 'WBDS03walkO03Fmkr.tx', 'WBDS03walkO04Fmkr.tx', 'WBDS03walkO05Fmkr.tx', 'WBDS03walkO06Fmkr.tx', 'WBDS03walkO07Fmkr.tx', 'WBDS03walkO08Fmkr.tx', 'WBDS03walkO09Fmkr.tx', 'WBDS03walk

### Fix filenames manually
It is easier to fix some filenames manually in the df_wbds dataframe, rather than trying to fix them programmatically. 

#### WBDS08walkO03grf.txt

In [15]:
# Display the row where the 'Subject' is 8 and the 'FileName' to spot the missing character 'S' in the 'FileName' column
df_wbds[(df_wbds['Subject'] == 8) & (df_wbds['FileName'] == 'WBDS08walkO03grf.txt')]

,Subject,FileName,AgeGroup,Age,Height,Mass,Gender,Dominance,LegLength,Static1,Static2,GaitSpeed(m/s),TreadHands,FP_RightFoot,FP_LeftFoot,Notes,BorgScale
1324,8,WBDS08walkO03grf.txt,Young,36,182.5,64.0,M,R,0.97,Yes,No,1.01,--,"FP4, FP1","FP3, FP2",--,12.0


In [16]:
# Address missing character 'S' in the 'FileName' column for Subject 8, where the 'FileName' is 'WBDS08walkO03grf.txt'. The correct filename should be 'WBDS08walkO03Sgrf.txt'.
# Replace the value of the 'FileName' column in df_wbds where the 'Subject' is 8 and the 'FileName' is 'WBDS08walkO03grf.txt' with 'WBDS08walkO03Sgrf.txt'
df_wbds.loc[(df_wbds['Subject'] == 8) & (df_wbds['FileName'] == 'WBDS08walkO03grf.txt'), 'FileName'] = 'WBDS08walkO03Sgrf.txt'

In [17]:
# Display the row where the 'Subject' is 8 and the 'FileName' is 'WBDS08walkO03Sgrf.txt' to confirm the change
df_wbds[(df_wbds['Subject'] == 8) & (df_wbds['FileName'] == 'WBDS08walkO03Sgrf.txt')]

,Subject,FileName,AgeGroup,Age,Height,Mass,Gender,Dominance,LegLength,Static1,Static2,GaitSpeed(m/s),TreadHands,FP_RightFoot,FP_LeftFoot,Notes,BorgScale
1324,8,WBDS08walkO03Sgrf.txt,Young,36,182.5,64.0,M,R,0.97,Yes,No,1.01,--,"FP4, FP1","FP3, FP2",--,12.0


#### Subject 3
Some filenames for subject 3 lacked the character "t" at the end of the filename.

In [18]:
df_wbds_not_in_filtered[df_wbds_not_in_filtered['Subject'] == 3]

,Subject,FileName,AgeGroup,Age,Height,Mass,Gender,Dominance,LegLength,Static1,Static2,GaitSpeed(m/s),TreadHands,FP_RightFoot,FP_LeftFoot,Notes,BorgScale
450,3,WBDS03walkT02mkr.tx,Young,33,179.3,75.85,M,L,0.94,Yes,No,0.54,No,--,--,--,8.0
451,3,WBDS03walkT03mkr.tx,Young,33,179.3,75.85,M,L,0.94,Yes,No,0.68,No,--,--,--,8.0
452,3,WBDS03walkT04mkr.tx,Young,33,179.3,75.85,M,L,0.94,Yes,No,0.83,No,--,--,--,8.0
453,3,WBDS03walkT05mkr.tx,Young,33,179.3,75.85,M,L,0.94,Yes,No,0.98,No,--,--,--,8.0
454,3,WBDS03walkT06mkr.tx,Young,33,179.3,75.85,M,L,0.94,Yes,No,1.12,No,--,--,--,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
530,3,WBDS03walkO09Fgrf.tx,Young,33,179.3,75.85,M,L,0.94,Yes,No,1.35,--,FP1,"FP3, FP2",--,8.0
531,3,WBDS03walkO10Fgrf.tx,Young,33,179.3,75.85,M,L,0.94,Yes,No,1.34,--,FP2,FP1,--,8.0
532,3,WBDS03walkO11Fgrf.tx,Young,33,179.3,75.85,M,L,0.94,Yes,No,1.37,--,--,"FP4, FP1",--,8.0
533,3,WBDS03walkO12Fgrf.tx,Young,33,179.3,75.85,M,L,0.94,Yes,No,1.36,--,"FP4, FP1","FP3, FP2",--,8.0


In [19]:
# Get the filenames from df_wbds_not_in_filtered where the 'Subject' is 3
fnames_S3 = df_wbds_not_in_filtered[df_wbds_not_in_filtered['Subject'] == 3]['FileName'].tolist()

In [20]:
# Add a character 't' at the end of each element in fnames_S3. Then, find and replace the values in the 'FileName' column of df_wbds with the updated values. For example, replace 'WBDS03walkT02mkr.tx' by 'WBDS03walkT02mkr.txt' and 'WBDS03walkT02grf.tx' by 'WBDS03walkT02grf.txt'; and so on.
for i in range(len(fnames_S3)):
    if fnames_S3[i].endswith('.tx'):
        new_fname = fnames_S3[i] + 't'
        df_wbds.loc[df_wbds['FileName'] == fnames_S3[i], 'FileName'] = new_fname

#### Subjects 43 to 51
The filenames for subjects 43 to 51 have two additional rows (V7 and V8) that need to be removed. 

In [21]:
# Drop rows from df_wbds that meet the criteria in df_wbds_not_in_filtered[df_wbds_not_in_filtered['Subject'] >= 43]['FileName']
df_wbds = df_wbds[~df_wbds['FileName'].isin(df_wbds_not_in_filtered[df_wbds_not_in_filtered['Subject'] >= 43]['FileName'])]

### Check if the number of files matches the number of files in the Figshare dataset.

In [22]:
# List of filenames that are in both filenames_txt and filenames_c3d
filenames_both = filenames_txt + filenames_c3d
# Find elements in filenames_txt and filenames_c3d that are not in df_wbds['FileName']
fnames_not_in_df_wbds = [f for f in filenames_both if f not in df_wbds['FileName'].values]

In [23]:
# Difference between the number of files in the Figshare dataset and the number of files in df_wbds
num_files_figshare = len(filenames_both)
num_files_df_wbds = len(df_wbds)
num_files_not_in_df_wbds = len(fnames_not_in_df_wbds)
diff_num_files = num_files_figshare - num_files_df_wbds 
print(f'Number of files in Figshare dataset: {num_files_figshare}')
print(f'Number of files in df_wbds: {num_files_df_wbds}')
print(f'Difference in number of files btw Figshare and df_wbds: {diff_num_files}')
print(f'Number of files not in df_wbds: {num_files_not_in_df_wbds}')
# if diff_num_files and num_files_not_in_df_wbds not equal to zero, then there are files in the Figshare dataset that are not in df_wbds. Print the list of files that are not in df_wbds.
if diff_num_files != 0 and num_files_not_in_df_wbds != 0:
    # Print in RED
    print('\033[91mThe filenames do not match between the Figshare dataset and df_wbds\033[0m')
else:
    # Print in GREEN
    print('\033[92mThe filenames match between the Figshare dataset and df_wbds. The number of filenames missing from df_wbds does not match the the difference between Figshare and df_wbds.\033[0m')

Number of files in Figshare dataset: 7203
Number of files in df_wbds: 6978
Difference in number of files btw Figshare and df_wbds: 225
Number of files not in df_wbds: 249
The filenames do not match between the Figshare dataset and df_wbds


As we can see, the difference between the number of files in the Figshare dataset and the number of files in the df_wbds is different than the filenames that are not listed in df_wbds. Most likely that is because some of the filenames are duplicated in the df_wbds. So, let's check if there are any duplicated filenames in the df_wbds dataframe.

In [24]:
# Find duplicates in df_wbds based on the 'FileName' column and display them
df_wbds_duplicated_fnames = df_wbds[df_wbds.duplicated(subset=['FileName'], keep=False)].sort_values(by=['FileName'])
df_wbds_duplicated_fnames

,Subject,FileName,AgeGroup,Age,Height,Mass,Gender,Dominance,LegLength,Static1,Static2,GaitSpeed(m/s),TreadHands,FP_RightFoot,FP_LeftFoot,Notes,BorgScale
535,3,WBDS03walkOCang.txt,Young,33,179.3,75.85,M,L,0.94,--,--,--,--,--,--,--,8.0
1192,7,WBDS03walkOCang.txt,Young,24,157.5,71.75,F,R,0.84,--,--,--,--,--,--,--,11.0
1193,7,WBDS03walkOCknt.txt,Young,24,157.5,71.75,F,R,0.84,--,--,--,--,--,--,--,11.0
536,3,WBDS03walkOCknt.txt,Young,33,179.3,75.85,M,L,0.94,--,--,--,--,--,--,--,8.0
537,3,WBDS03walkOFang.txt,Young,33,179.3,75.85,M,L,0.94,--,--,--,--,--,--,--,8.0
1194,7,WBDS03walkOFang.txt,Young,24,157.5,71.75,F,R,0.84,--,--,--,--,--,--,--,11.0
538,3,WBDS03walkOFknt.txt,Young,33,179.3,75.85,M,L,0.94,--,--,--,--,--,--,--,8.0
1195,7,WBDS03walkOFknt.txt,Young,24,157.5,71.75,F,R,0.84,--,--,--,--,--,--,--,11.0
1196,7,WBDS03walkOSang.txt,Young,24,157.5,71.75,F,R,0.84,--,--,--,--,--,--,--,11.0
539,3,WBDS03walkOSang.txt,Young,33,179.3,75.85,M,L,0.94,--,--,--,--,--,--,--,8.0


We'll fix the duplicates manually, and then check if the number of files and their names are consistent with the files in the Figshare dataset.

In [25]:
# Subject 7 has many wrong filenames. Let's check the filenames for Subject 7 in df_wbds_duplicated_fnames
df_duplicates_s7 = df_wbds_duplicated_fnames[df_wbds_duplicated_fnames['Subject'] == 7]

In [26]:
# Replace the values of the 'FileName' column in df_wbds where the 'Subject' is 7 and the 'FileName' is in df_duplicates_s7. Basically, we'll replace the 6th character of the 'FileName' with '7' instead of '3' such that it reflects the correct subject number. For example, replace 'WBDS03walkOCang.txt.txt' by 'WBDS07walkOCang.txt.txt'; and so on.
for i in range(len(df_duplicates_s7)):
    old_fname = df_duplicates_s7.iloc[i]['FileName']
    new_fname = old_fname[:5] + '7' + old_fname[6:]
    # print old_fname and new_fname to check if the replacement is correct
    # print(f'Old filename: {old_fname}, New filename: {new_fname}')
    # Replace 'Filename' in df_wbds where 'FileName' is old_fname and 'Subject' is 7 with new_fname
    df_wbds.loc[(df_wbds['FileName'] == old_fname) & (df_wbds['Subject'] == 7), 'FileName'] = new_fname

Drop duplicates in df_wbds that matches df_wbds_duplicated_fnames when the 'Subject' is 6.

In [27]:
# Drop duplicates in df_wbds that matches df_wbds_duplicated_fnames when the 'Subject' is 6.
df_wbds = df_wbds[~((df_wbds['FileName'].isin(df_wbds_duplicated_fnames['FileName'])) & (df_wbds['Subject'] == 6))]

Check duplicates in df_wbds that matches df_wbds_duplicated_fnames when the 'Subject' is 13.
The subject 13 has two files with the same name, but the other columns are different which indicates that they are different files.

In [28]:
# Display rows in df_wbds that matches the pattern '.txt' and the 'Subject' is 13. The subject 13 has two files with the same name, but the other columns are different which indicates that they are different files.
df_wbds[
    df_wbds['FileName'].str.contains(r'walkO.*F.*mkr.*\.txt$', regex=True, na=False)
    & (df_wbds['Subject'] == 13)
]

,Subject,FileName,AgeGroup,Age,Height,Mass,Gender,Dominance,LegLength,Static1,Static2,GaitSpeed(m/s),TreadHands,FP_RightFoot,FP_LeftFoot,Notes,BorgScale
2122,13,WBDS13walkO01Fmkr.txt,Young,30,171.0,95.4,M,R,0.85,Yes,No,1.47,--,FP1,"FP4_3, FP2",--,13.0
2123,13,WBDS13walkO02Fmkr.txt,Young,30,171.0,95.4,M,R,0.85,Yes,No,1.2,--,FP1,"FP3, FP2",--,13.0
2124,13,WBDS13walkO03Fmkr.txt,Young,30,171.0,95.4,M,R,0.85,Yes,No,1.22,--,--,"FP4, FP1",--,13.0
2125,13,WBDS13walkO04Fmkr.txt,Young,30,171.0,95.4,M,R,0.85,Yes,No,1.33,--,--,"FP4, FP1_2",--,13.0
2126,13,WBDS13walkO05Fmkr.txt,Young,30,171.0,95.4,M,R,0.85,Yes,No,1.26,--,FP2,"FP4, FP1",--,13.0
2127,13,WBDS13walkO06Fmkr.txt,Young,30,171.0,95.4,M,R,0.85,Yes,No,1.24,--,FP2,"FP4, FP1",--,13.0
2128,13,WBDS13walkO07Fmkr.txt,Young,30,171.0,95.4,M,R,0.85,Yes,No,1.31,--,--,"FP4, FP1",--,13.0
2129,13,WBDS13walkO07Fmkr.txt,Young,30,171.0,95.4,M,R,0.85,Yes,No,1.29,--,FP2,"FP4, FP1",--,13.0
2130,13,WBDS13walkO09Fmkr.txt,Young,30,171.0,95.4,M,R,0.85,Yes,No,1.27,--,FP2,"FP4, FP1",--,13.0
2131,13,WBDS13walkO10Fmkr.txt,Young,30,171.0,95.4,M,R,0.85,Yes,No,1.23,--,FP1,"FP3, FP2",--,13.0


In [29]:
# Found the problematic row (index 2129). Replace the value of the 'FileName' column in df_wbds where the 'Subject' is 13 and the 'FileName' is 'WBDS13walkO07Fmkr.txt' with 'WBDS13walkO08Fmkr.txt'. Bear in mind there are two rows with the same 'FileName' but different values in the other columns. The row to be replaced is with index 2129.
# Use the index to locate the row and replace the value of the 'FileName' column
df_wbds.loc[2129, 'FileName'] = 'WBDS13walkO08Fmkr.txt'

### Re-check on metadata df_wbds to ensure that the number of files and their names are consistent with the files in the Figshare dataset.

In [30]:
# Find elements in filenames_txt and filenames_c3d that are not in df_wbds['FileName']
fnames_not_in_df_wbds = [f for f in filenames_both if f not in df_wbds['FileName'].values]

In [31]:
# Difference between the number of files in the Figshare dataset and the number of files in df_wbds
num_files_figshare = len(filenames_both)
num_files_df_wbds = len(df_wbds)
num_files_not_in_df_wbds = len(fnames_not_in_df_wbds)
diff_num_files = num_files_figshare - num_files_df_wbds 
print(f'Number of files in Figshare dataset: {num_files_figshare}')
print(f'Number of files in df_wbds: {num_files_df_wbds}')
print(f'Difference in number of files btw Figshare and df_wbds: {diff_num_files}')
print(f'Number of files not in df_wbds: {num_files_not_in_df_wbds}')
# if diff_num_files and num_files_not_in_df_wbds not equal to zero, then there are files in the Figshare dataset that are not in df_wbds. Print the list of files that are not in df_wbds.
if diff_num_files != 0 and num_files_not_in_df_wbds != 0:
    # Print in RED
    print('\033[91mThe filenames do not match between the Figshare dataset and df_wbds\033[0m')
else:
    # Print in GREEN
    print('\033[92mThe filenames match between the Figshare dataset and df_wbds\033[0m')

Number of files in Figshare dataset: 7203
Number of files in df_wbds: 6976
Difference in number of files btw Figshare and df_wbds: 227
Number of files not in df_wbds: 227
The filenames do not match between the Figshare dataset and df_wbds


Now the difference in the number of files in the Figshare dataset and the number of files in the df_wbds is consistent with the filenames that are not listed in df_wbds. Therefore, we just need to resolve the filenames that are not listed in df_wbds, and then we can create the WBDSinfo.xlsx file.

In [32]:
# Filter df_wbds to only include rows where the 'FileName' exists in filenames_txt or filenames_c3d
df_wbds_filtered2 = df_wbds[df_wbds['FileName'].isin(filenames_txt) | df_wbds['FileName'].isin(filenames_c3d)]

In [33]:
# Filter df_wbds to only include rows where the 'FileName' are not in df_wbds_filtered2
df_wbds_not_in_filtered2 = df_wbds[~df_wbds['FileName'].isin(df_wbds_filtered2['FileName'])]
# Print if df_wbds_not_in_filtered2 is empty or not
if df_wbds_not_in_filtered2.empty:
    print('All files in df_wbds are accounted for in the txt and c3d files.')
else:
    print('There are files in df_wbds that are not accounted for in the txt and c3d files.')

All files in df_wbds are accounted for in the txt and c3d files.


### Address filenames not in df_wbds
We know that most of the 227 files missing are the ones from the subject 43 onwards. Subject 6 have some files missing as well.

Subject 6

In [34]:
# Insert a row in df_wbds for the missing file 'WBDS06walkO06Sgrf.txt' for Subject 6. The new row should have the same values as the row for Subject 6 and FileName 'WBDS06walkO06S.c3d', except for the 'FileName' column which should be updated to the new filename.
new_row = df_wbds[df_wbds['FileName'] == 'WBDS06walkO06S.c3d'].iloc[0].copy()
new_row['FileName'] = 'WBDS06walkO06Sgrf.txt'
df_wbds = pd.concat(
    [df_wbds, new_row.to_frame().T],
    ignore_index=True
)

In [35]:
# Insert a row in df_wbds for the missing file 'WBDS06walkO06Smkr.txt' for Subject 6. The new row should have the same values as the row for Subject 6 and FileName 'WBDS06walkO06S.c3d', except for the 'FileName' column which should be updated to the new filename.
new_row = df_wbds[df_wbds['FileName'] == 'WBDS06walkO06S.c3d'].iloc[0].copy()
new_row['FileName'] = 'WBDS06walkO06Smkr.txt'
df_wbds = pd.concat(
    [df_wbds, new_row.to_frame().T],
    ignore_index=True
)

### Address Subjects 43 onwards. 
need to copy the rows based on the c3d files and create for txt files (ang, grf, knt, mkr). The new rows should have the same values as the row for the subject and FileName 'WBDSxxwalkOxxS.c3d', except for the 'FileName' column which should be updated to the new filename.

In [36]:
# List of subjects to be fixed 43 to 51
subjects_to_fix = [43, 44, 45, 46, 47, 48, 49, 50, 51]

In [37]:
# Add rows to df_wbds for the missing files for subjects 43 to 51. The new rows should have the same values as the row for the subject and FileName 'WBDSxxstatic1.c3d' or 'WBDSxxwalkTxx.c3d', except for the 'FileName' column which should be updated to the new filename.
extension = ['ang', 'grf', 'knt', 'mkr']
for subject in subjects_to_fix:
    # Get the 'FileName' values for the subject in df_wbds
    file_name_static = df_wbds[df_wbds['Subject'] == subject]['FileName'].tolist()
    # Create a 'static' filename with the same name pattern as the element of file_names containing 'static' but the extension '.txt' instead of '.c3d'. For example, if the element of file_names is 'WBDS43static1.c3d', then the new filename would be 'WBDS43static1.txt'.
    static_filenames = [f.replace('.c3d', '.txt') for f in file_name_static if 'static' in f]
    # Duplicate the row where the 'Subject' is subject and the 'FileName' is 'WBDSxxstatic1.c3d' and update the 'FileName' column to the new filename in static_filenames. Then, append the new row to df_wbds. Bear in mind static_filenames has updated filenames with the extension '.txt' instead of '.c3d', so we need to get the original filename with the extension '.c3d' to find the row in df_wbds. For example, if the new filename is 'WBDS43static1.txt', then the original filename would be 'WBDS43static1.c3d'.
    new_static_row = df_wbds[df_wbds['FileName'] == static_filenames[0].replace('.txt', '.c3d')].iloc[0].copy() # static_filenames[0] is the new filename with the extension '.txt', so we need to get the original filename with the extension '.c3d' to find the row in df_wbds. For example, if the new filename is 'WBDS43static1.txt', then the original filename would be 'WBDS43static1.c3d'.
    new_static_row['FileName'] = static_filenames[0]
    df_wbds = pd.concat([df_wbds, new_static_row.to_frame().T], ignore_index=True)

    # For the remaining filenames where the pattern 'walkT' is present, create four filenames for each 'c3d' file, replacing the extension with '.txt' and appending 'mkr', 'grf', 'ang', and 'knt' to the base filename. For example, if the element of file_names is 'WBDS43walkT01.c3d', then the new filenames would be 'WBDS43walkT01mkr.txt', 'WBDS43walkT01grf.txt', 'WBDS43walkT01ang.txt', and 'WBDS43walkT01knt.txt'. 
    # Delete the elements of file_names that contain 'static'
    file_names = [f for f in file_name_static if 'static' not in f]
    # Copy the rows of df_wbds where the 'Subject' is subject and the 'FileName' is in file_names. For each copied row, create four new rows with the same values as the copied row, except for the 'FileName' column which should be updated to the new filenames. The new filenames should be created by replacing the extension with '.txt' and appending 'mkr', 'grf', 'ang', and 'knt' to the base filename. For example, if the element of file_names is 'WBDS43walkT01.c3d', then the new filenames would be 'WBDS43walkT01mkr.txt', 'WBDS43walkT01grf.txt', 'WBDS43walkT01ang.txt', and 'WBDS43walkT01knt.txt'.
    for f in file_names:
        if 'walkT' in f:
            base_name = f.replace('.c3d', '')
            for ext in extension:
                new_row = df_wbds[df_wbds['FileName'] == f].iloc[0].copy()
                new_row['FileName'] = base_name + ext + '.txt'
                df_wbds = pd.concat([df_wbds, new_row.to_frame().T], ignore_index=True)

## Final check on metadata df_wbds to ensure that the number of files and their names are consistent with the files in the Figshare dataset.

In [38]:
# Check if the elements of filenames_both are in df_wbds['FileName'] and return a list of the elements that are not in df_wbds['FileName']
fnames_not_in_df_wbds = [f for f in filenames_both if f not in df_wbds['FileName'].values]
# Print the output in RED if the list is not empty, otherwise print in GREEN
if fnames_not_in_df_wbds:
    print('\033[91mThe following filenames are not in df_wbds:\033[0m')
    print(fnames_not_in_df_wbds)
else:
    print('\033[92mAll filenames are in df_wbds.\033[0m')

All filenames are in df_wbds.


In [39]:
# Sort df_wbds by 'Subject' and 'FileName'
df_wbds = df_wbds.sort_values(by=['Subject', 'FileName']).reset_index(drop=True)

In [ ]:
# Export df_wbds to a csv file in path2 with the name 'WBDSinfo_M_reproduced.csv'
# df_wbds.to_csv(path2 / 'WBDSinfo.csv', index=False)